# **Preprocessing of the data**

In [38]:
from calendar import error

import pandas as pd
import numpy as np
import os

In [39]:
data_dir = "../data/processed/"
listdir = os.listdir(data_dir)
listdir

['TEST_8_processed.csv',
 'TEST_15_processed.csv',
 'TEST_11_processed.csv',
 'TEST_2_processed.csv',
 'TEST_6_processed.csv',
 'TEST_1_processed.csv',
 'TEST_5_processed.csv',
 'TEST_16_processed.csv',
 'TEST_12_processed.csv',
 'TEST_4_processed.csv',
 'TEST_13_processed.csv',
 'TEST_17_processed.csv',
 'TEST_10_processed.csv',
 'TEST_14_processed.csv',
 'TEST_7_processed.csv',
 'TEST_3_processed.csv',
 'TEST_9_processed.csv']

In [40]:
BATTERY_METADATA = {
    "TEST_1_processed":  {"capacity": 85,    "charged": 85,    "type": "b5"},
    "TEST_2_processed":  {"capacity": 81.28, "charged": 81.28, "type": "b1"},
    "TEST_3_processed":  {"capacity": 85,    "charged": 85,    "type": "b5"},
    "TEST_4_processed":  {"capacity": 85,    "charged": 85,    "type": "b2"},
    "TEST_5_processed":  {"capacity": 88.81, "charged": 88.81, "type": "b2"},
    "TEST_6_processed":  {"capacity": 81.84, "charged": 81.84, "type": "b1"},
    "TEST_7_processed":  {"capacity": 81.84, "charged": 36,    "type": "b1"},
    "TEST_8_processed":  {"capacity": 88.81, "charged": 27,    "type": "b2"},
    "TEST_9_processed":  {"capacity": 85,    "charged": 80,    "type": "tn1"},
    "TEST_10_processed": {"capacity": 85,    "charged": 54,    "type": "tn1"},
    "TEST_11_processed": {"capacity": 85,    "charged": 85,    "type": "b5"},
    "TEST_12_processed": {"capacity": 85,    "charged": 67,    "type": "b5"},
    "TEST_13_processed": {"capacity": 85,    "charged": 85,    "type": "b5"},
    "TEST_14_processed": {"capacity": 88.83, "charged": 52,    "type": "b3"},
    "TEST_15_processed": {"capacity": 88.35, "charged": 70,    "type": "b3"},
    "TEST_16_processed": {"capacity": 88.35, "charged": 61,    "type": "b3"},
    "TEST_17_processed": {"capacity": 88.35, "charged": 88.35, "type": "b3"},
}

In [41]:
DATASET = pd.DataFrame()

In [42]:
for i in listdir:
    for j in BATTERY_METADATA.keys():
        k = j+'.csv'
        if i==k:
            dataset = pd.read_csv(data_dir+i)
            dataset['type'] = BATTERY_METADATA[j]['type']
            dataset['capacity'] = BATTERY_METADATA[j]['capacity']
            dataset['charged'] = BATTERY_METADATA[j]['charged']
            DATASET = pd.concat([DATASET,dataset],ignore_index=True)

DATASET

/var/folders/kc/jh1xkqxn3vdd35xtydf9ktqc0000gn/T/ipykernel_54460/1277646566.py:9: FutureWarning: The behavior of DataFrame concatenation with empty or all-NA entries is deprecated. In a future version, this will no longer exclude empty or all-NA columns when determining the result dtypes. To retain the old behavior, exclude the relevant entries before the concat operation.
  DATASET = pd.concat([DATASET,dataset],ignore_index=True)


,Current,Voltage,Ah Out,Cumulative Actual Disch Ah,Power,Remaining Capacity,Time to Depletion,type,capacity,charged
0,9.36,11.84,0.156000,0.156000,110.8224,26.844000,10324.615385,b2,88.81,27.0
1,9.34,11.84,0.155667,0.311667,110.5856,26.688333,10286.723769,b2,88.81,27.0
2,9.34,11.83,0.155667,0.467333,110.4922,26.532667,10226.723769,b2,88.81,27.0
3,7.14,11.88,0.119000,0.586333,84.8232,26.413667,13317.815126,b2,88.81,27.0
4,7.13,11.88,0.118833,0.705167,84.7044,26.294833,13276.493689,b2,88.81,27.0
...,...,...,...,...,...,...,...,...,...,...
5432,6.69,9.74,0.111500,75.816500,65.1606,4.183500,2251.210762,tn1,85.00,80.0
5433,6.69,9.75,0.111500,75.928000,65.2275,4.072000,2191.210762,tn1,85.00,80.0
5434,6.69,9.75,0.111500,76.039500,65.2275,3.960500,2131.210762,tn1,85.00,80.0
5435,6.01,9.83,0.100167,76.139667,59.0783,3.860333,2312.346090,tn1,85.00,80.0


In [43]:
with open("../new_code/DATASET.csv", "w") as f:
    pd.DataFrame.to_csv(DATASET, f, index=False)

# **Testing of Simulation**

In [104]:
import numpy as np
import time
from datetime import datetime
import xgboost as xgb
from sklearn.preprocessing import OneHotEncoder
from sklearn.pipeline import Pipeline

In [105]:
MODEL_PATH = "../models/battery_xgboost_model.json"
TIME_STEPS = 10

In [106]:
model = xgb.XGBRegressor()
model.load_model(MODEL_PATH)

In [107]:
base_current = 15
base_voltage = 12.4
cumulative_ah = 0
predictions = []
timestamps = []
buffer = []
start_time = datetime.now()
battery_type = {
    "B1":81.28,
    "B2":85,
    "B3":88.35,
    "TN1":85,
    "B5":85
}
battery_order = {
    "B1":8,
    "B2":9,
    "B3":10,
    "TN1":12,
    "B5":11
}

In [109]:
n = 3
while True:
    battery_capacity = float(battery_type[list(battery_type.keys())[n]])
    battery_code = list(battery_type.keys())[n]
    battery_order_code = battery_order[battery_code]
    t = (datetime.now() - start_time).seconds

    current = base_current + 3 * np.sin(t / 10) + np.random.normal(0, 1)
    voltage = base_voltage + 0.05 * np.sin(t / 15) + np.random.normal(0, 0.01)
    current = max(5, min(current, 50))
    voltage = max(9.0, min(voltage, 13.0))

    if voltage < 9.4:
        break

    ah_out = current / 3600
    cumulative_ah += ah_out
    power = current * voltage
    remaining = max(battery_capacity - cumulative_ah, 0)

    base_row = [
        float(current),
        float(voltage),
        float(ah_out),
        float(cumulative_ah),
        float(power),
        float(remaining),
        float(battery_capacity),
        float(battery_capacity),
        0.0,
        0.0,
        0.0,
        0.0,
        0.0
    ]
    base_row[battery_order_code] = 1.0

    buffer.append(base_row)

    if len(buffer) >= TIME_STEPS:
        X_input = np.array(buffer, dtype=np.float32)
        y_pred = model.predict(X_input)
        discharge_percent = np.clip(y_pred[0], 0, 100)
        time_remaining = (discharge_percent / 100) * (battery_capacity * 3600 / base_current)

        print("⚡ Current (A)", f"{current:.2f}")
        print("🔋 Voltage (V)", f"{voltage:.2f}")
        h = int(time_remaining // 3600)
        m = int((time_remaining % 3600) // 60)
        s = int(time_remaining % 60)
        print("⏳ Remaining Time", f"{h:02d}:{m:02d}:{s:02d}")
        print("🕒 Running Time", str(datetime.now() - start_time).split('.')[0])

        predictions.append(time_remaining / 3600)
        timestamps.append(datetime.now().strftime('%H:%M:%S'))

        buffer.pop(0)

    time.sleep(1)

⚡ Current (A) 16.42
🔋 Voltage (V) 12.38
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:32
⚡ Current (A) 16.28
🔋 Voltage (V) 12.39
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:33
⚡ Current (A) 15.10
🔋 Voltage (V) 12.39
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:34
⚡ Current (A) 13.05
🔋 Voltage (V) 12.40
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:35
⚡ Current (A) 14.05
🔋 Voltage (V) 12.41
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:36
⚡ Current (A) 14.30
🔋 Voltage (V) 12.40
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:37
⚡ Current (A) 14.91
🔋 Voltage (V) 12.41
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:38
⚡ Current (A) 13.10
🔋 Voltage (V) 12.41
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:39
⚡ Current (A) 14.72
🔋 Voltage (V) 12.41
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:40
⚡ Current (A) 11.72
🔋 Voltage (V) 12.42
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:41
⚡ Current (A) 11.83
🔋 Voltage (V) 12.42
⏳ Remaining Time 05:40:00
🕒 Running Time 0:01:42
⚡ Current (A) 13.21
🔋

KeyboardInterrupt: 

In [101]:
battery_order_code

12